In [74]:
!pip install google-generativeai scikit-learn numpy

In [75]:
import google.generativeai as genai
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [76]:
genai.configure(api_key="YOUR_REAL_API_KEY")


In [77]:
model = genai.GenerativeModel("gemini-pro")

In [78]:
documents = [
    "Artificial Intelligence is the simulation of human intelligence in machines.",
    "Machine Learning is a subset of Artificial Intelligence.",
    "Deep Learning uses neural networks.",
    "Natural Language Processing helps machines understand language.",
    "Companies improve profit growth using innovation, cost reduction, and market expansion."
]


In [79]:
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)


In [80]:
def retrieve(query, top_k=2):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, doc_vectors)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]
    retrieved_docs = [documents[i] for i in top_indices]
    return "\n".join(retrieved_docs), max(similarities)


In [81]:
def expand_query(query):
    print("\n--- R3: Query Expansion ---")
    try:
        response = model.generate_content(f"Rewrite this query clearly:\n{query}")
        return response.text
    except Exception as e:
        print("R3 Failed → Using original query")
        return query

In [82]:
def generate_answer(query, context):
    print("\n--- R4: Context-Based Answer ---")
    try:
        prompt = f"""
        Answer ONLY using this context.

        Context:
        {context}

        Question:
        {query}
        """
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print("R4 Failed → Returning fallback answer")
        return f"Based on retrieved context:\n{context}"

In [83]:
def failover(score, threshold=0.3):
    print("\n--- R5: Failover Check ---")
    print("Similarity Score:", score)

    if score < threshold:
        print("R5 Status: FAIL")
        return False
    else:
        print("R5 Status: PASS")
        return True

In [84]:
def multi_stage_rag(query):

    print("\n--- R1: Strict Retrieval ---")
    context, score = retrieve(query)
    print("Similarity Score:", score)

    if not failover(score):
        return "R5 Triggered: Not enough relevant data."

    expanded_query = expand_query(query)

    print("\n--- R2: Broad Retrieval ---")
    context, score = retrieve(expanded_query)
    print("Similarity Score:", score)

    if not failover(score):
        return "R5 Triggered After Expansion."

    answer = generate_answer(query, context)
    return answer

In [85]:
print(multi_stage_rag("company profit growth strategy"))


--- R1: Strict Retrieval ---
Similarity Score: 0.4264014327112209

--- R5: Failover Check ---
Similarity Score: 0.4264014327112209
R5 Status: PASS

--- R3: Query Expansion ---


R3 Failed → Using original query

--- R2: Broad Retrieval ---
Similarity Score: 0.4264014327112209

--- R5: Failover Check ---
Similarity Score: 0.4264014327112209
R5 Status: PASS

--- R4: Context-Based Answer ---
R4 Failed → Returning fallback answer
Based on retrieved context:
Companies improve profit growth using innovation, cost reduction, and market expansion.
Natural Language Processing helps machines understand language.


Lab Exercise for Students

1. Make your answer include citations, e.g.:

> “RAG reduces hallucinations by grounding answers in documents [rag_definition_chunk_1].”

2. Introduce a score threshold: if a retrieved chunk’s similarity score falls below this value, it should be excluded from the context. Then simulate a scenario where none of the retrieved chunks meet the threshold, demonstrating how the system should behave when no context is sufficiently relevant.

3. Experiment with different retrieval depths by varying k (e.g., k = 1, 3, 5). Observe how increasing or reducing the number of retrieved chunks affects both the quality of the Gemini-generated answer and the citations used. Identify cases where a higher k introduces noise versus cases where a lower k misses important context.”
